# 002 · Locks, Events and Queues

> **Standard Library › Concurrency** · Python 3.14

As soon as threads share data, you need synchronisation. The `threading` module provides the same primitives you know from `System.Threading` — locks, events, semaphores, barriers — and the `queue` module provides thread-safe queues, the backbone of producer/consumer designs.

**In this notebook you will learn:**

- why `counter += 1` is a race condition even with the GIL, and how to fix it with `Lock`
- `Lock` vs `RLock` and how both relate to C#'s `lock` statement / `Monitor`
- signalling with `Event` (≈ `ManualResetEvent`) and `Condition` (≈ `Monitor.Wait/Pulse`)
- limiting concurrency with `Semaphore` (≈ `SemaphoreSlim`) and synchronising phases with `Barrier`
- producer/consumer with `queue.Queue` (≈ `BlockingCollection<T>` / `Channel<T>`): `task_done`/`join`, sentinels, `shutdown()`
- `LifoQueue`, `PriorityQueue` and `SimpleQueue`

## .NET mental model

| .NET | Python |
|---|---|
| `lock (obj) { ... }` / `Monitor.Enter/Exit` | `with lock:` (`threading.Lock` / `RLock`) |
| `System.Threading.Lock` (C# 13) | `threading.Lock` |
| `ManualResetEvent(Slim)` | `threading.Event` |
| `Monitor.Wait` / `Pulse` / `PulseAll` | `threading.Condition`: `wait` / `notify` / `notify_all` |
| `SemaphoreSlim` | `threading.Semaphore` / `BoundedSemaphore` |
| `System.Threading.Barrier` | `threading.Barrier` |
| `BlockingCollection<T>` / `Channel<T>` | `queue.Queue` |
| `ConcurrentStack<T>` | `queue.LifoQueue` |
| `PriorityQueue<TElement, TPriority>` (not thread-safe) | `queue.PriorityQueue` (thread-safe) |
| `Interlocked.Increment` | no equivalent — use a `Lock` |

All of these primitives (except queues) support the `with` statement, which acquires on entry and releases on exit — exactly like C#'s `lock` block or `using` scope.

## The race condition behind `counter += 1`

`counter += 1` looks atomic but is several bytecode steps: **load** the value, **add**, **store** it back. The GIL can switch threads between any two steps:

In [ ]:
import dis

counter = 0
def increment():
    global counter
    counter += 1

dis.dis(increment)

If thread A loads `0`, then thread B loads `0`, both store `1` — one increment is **lost**.

Honest caveat: on a standard (GIL) build, CPython only switches threads at certain points, so a naive loop of `counter += 1` in several threads *often* produces the right answer and the bug may not show up in a demo — it appears under load, on a different Python version, or on a free-threaded `3.14t` build. Don't rely on luck.

To make the lost update **deterministic** for this demo, we force both threads to read before either writes, using a `Barrier` (explained later in this notebook):

In [ ]:
import threading

counter = 0
both_have_read = threading.Barrier(2)

def unsafe_increment():
    global counter
    value = counter          # 1. load
    both_have_read.wait()    #    (force the unlucky interleaving)
    counter = value + 1      # 2. add + store

ts = [threading.Thread(target=unsafe_increment) for _ in range(2)]
for t in ts: t.start()
for t in ts: t.join()
print("expected 2, got", counter)   # got 1 — an update was lost

## `Lock` and the `with` statement

```csharp
// C#
private readonly Lock _gate = new();   // C# 13; or: private readonly object _gate = new();
lock (_gate)
{
    _counter++;
}
```

`threading.Lock` is a mutual-exclusion lock. Use it with `with` so it is always released, even if an exception occurs:

In [ ]:
counter = 0
lock = threading.Lock()

def safe_increment(times):
    global counter
    for _ in range(times):
        with lock:              # acquire ... release (C#: lock (_gate) { ... })
            counter += 1

ts = [threading.Thread(target=safe_increment, args=(10_000,)) for _ in range(4)]
for t in ts: t.start()
for t in ts: t.join()
print(counter)   # always 40000

You can also call `acquire()`/`release()` explicitly, and `acquire` takes `blocking=` and `timeout=` — the equivalent of `Monitor.TryEnter(obj, timeout)`:

In [ ]:
lock = threading.Lock()
print(lock.acquire())                 # True — we hold it now
print(lock.acquire(timeout=0.1))      # False — already held (waited 0.1s)
print(lock.locked())                  # True
lock.release()
print(lock.locked())                  # False

## `RLock` — the reentrant lock

⚠️ **Gotcha:** C#'s `lock`/`Monitor` is **reentrant**: the owning thread may enter the same lock again. Python's `Lock` is **not** — a thread that acquires a `Lock` it already holds blocks forever (deadlock). Use `threading.RLock` when a method holding the lock calls another method that takes the same lock.

In [ ]:
class Account:
    def __init__(self):
        self._lock = threading.RLock()   # try Lock() here and deposit_twice() would deadlock
        self.balance = 0

    def deposit(self, amount):
        with self._lock:
            self.balance += amount

    def deposit_twice(self, amount):
        with self._lock:                 # outer acquire
            self.deposit(amount)         # inner acquire by the SAME thread — OK with RLock
            self.deposit(amount)

acct = Account()
acct.deposit_twice(50)
print(acct.balance)   # 100

Seeing the non-reentrant behaviour safely (non-blocking acquire instead of a real deadlock):

In [ ]:
plain = threading.Lock()
with plain:
    print("Lock re-acquire: ", plain.acquire(blocking=False))   # False — would deadlock if blocking

reentrant = threading.RLock()
with reentrant:
    print("RLock re-acquire:", reentrant.acquire(blocking=False))  # True
    reentrant.release()                                          # release as many times as acquired

## `Event` — a one-way signal

```csharp
// C#
var ready = new ManualResetEventSlim(false);
// worker:  ready.Wait();
// main:    ready.Set();
```

An `Event` is a boolean flag threads can wait on. `set()` wakes all waiters, `clear()` resets it, `wait(timeout)` blocks until set (and returns `True`) or the timeout expires (returns `False`). It behaves like a **manual**-reset event.

In [ ]:
import time

start_signal = threading.Event()
log = []

def runner(name):
    start_signal.wait()                  # block until main says go
    log.append(name)

ts = [threading.Thread(target=runner, args=(n,)) for n in ["alice", "bob", "carol"]]
for t in ts: t.start()

time.sleep(0.05)
print("before set:", log)                # [] — everyone is waiting
start_signal.set()                       # release all waiters at once
for t in ts: t.join()
print("after set: ", sorted(log))

**Tip:** the idiomatic way to stop a background worker is a "stop" event plus `event.wait(interval)` as the sleep. Unlike `time.sleep`, it wakes up **immediately** when the event is set — the Python version of a `CancellationToken`.

In [ ]:
stop = threading.Event()
ticks = []

def poller():
    while not stop.wait(0.02):           # wait 20ms OR until stop is set
        ticks.append("tick")

t = threading.Thread(target=poller)
t.start()
time.sleep(0.15)
stop.set()                               # request cancellation
t.join()
print("stopped cleanly; ticked at least once:", len(ticks) > 0)

## `Condition` — wait until some state is true

```csharp
// C#
lock (_gate)
{
    while (_items.Count == 0) Monitor.Wait(_gate);
    var item = _items.Dequeue();
}
// producer: lock (_gate) { _items.Enqueue(x); Monitor.Pulse(_gate); }
```

A `Condition` combines a lock with wait/notify. `wait_for(predicate)` loops for you (handling spurious wake-ups), like the `while (...) Monitor.Wait(...)` pattern.

In [ ]:
from collections import deque

items = deque()
cond = threading.Condition()
consumed = []

def consumer():
    for _ in range(3):
        with cond:
            cond.wait_for(lambda: len(items) > 0)   # releases the lock while waiting
            consumed.append(items.popleft())

c = threading.Thread(target=consumer)
c.start()
for x in ["a", "b", "c"]:
    time.sleep(0.02)
    with cond:
        items.append(x)
        cond.notify()                                  # C#: Monitor.Pulse
c.join()
print(consumed)

In practice you rarely need a raw `Condition`: a `queue.Queue` (below) already implements this pattern for you.

## `Semaphore` — limit concurrency

```csharp
// C#
var gate = new SemaphoreSlim(2);
await gate.WaitAsync();
try { /* at most 2 at a time */ } finally { gate.Release(); }
```

A `Semaphore(n)` lets up to `n` threads in at once. `BoundedSemaphore` additionally raises `ValueError` if you release more times than you acquired — prefer it, it catches bugs.

In [ ]:
gate = threading.BoundedSemaphore(2)
active = 0
max_active = 0
stats_lock = threading.Lock()

def limited_job():
    global active, max_active
    with gate:                               # at most 2 threads inside
        with stats_lock:
            active += 1
            max_active = max(max_active, active)
        time.sleep(0.05)
        with stats_lock:
            active -= 1

ts = [threading.Thread(target=limited_job) for _ in range(6)]
for t in ts: t.start()
for t in ts: t.join()
print("max concurrent jobs:", max_active)    # 2

try:
    gate.release()                           # one release too many
except ValueError as e:
    print(f"{type(e).__name__}: {e}")

## `Barrier` — wait until everyone arrives

```csharp
// C#
var barrier = new Barrier(3);
barrier.SignalAndWait();
```

`Barrier(parties)` blocks each thread in `wait()` until `parties` threads have arrived, then releases them all together — useful for phased algorithms. `wait()` returns a distinct index `0..parties-1` to each thread.

In [ ]:
barrier = threading.Barrier(3)
phases = []
phase_lock = threading.Lock()

def worker(name):
    with phase_lock: phases.append(("phase 1", name))
    barrier.wait()                       # nobody starts phase 2 until all finished phase 1
    with phase_lock: phases.append(("phase 2", name))

ts = [threading.Thread(target=worker, args=(n,)) for n in ["a", "b", "c"]]
for t in ts: t.start()
for t in ts: t.join()

print([p for p, _ in phases])            # all phase 1 entries come before any phase 2

## Producer/consumer with `queue.Queue`

```csharp
// C#
using var queue = new BlockingCollection<int>(boundedCapacity: 10);
// producer: queue.Add(x); ... queue.CompleteAdding();
// consumer: foreach (var x in queue.GetConsumingEnumerable()) ...
// or System.Threading.Channels: Channel.CreateBounded<int>(10)
```

`queue.Queue` is a thread-safe FIFO with blocking `put()`/`get()`. Pass `maxsize` to make it bounded — `put()` then blocks when the queue is full (back-pressure).

Two ways to know when the work is done:

- **`task_done()` + `join()`**: each consumer calls `q.task_done()` after processing an item; `q.join()` blocks until every item that was `put` has been marked done.
- **sentinel**: put a special value (often `None`) per consumer meaning "no more work" — the equivalent of `CompleteAdding()`.

In [ ]:
import queue

q = queue.Queue(maxsize=5)
processed = []
done_lock = threading.Lock()

def consumer(name):
    while True:
        item = q.get()                 # blocks until an item is available
        try:
            if item is None:           # sentinel: stop this consumer
                return
            with done_lock:
                processed.append((item, item * item))
        finally:
            q.task_done()              # mark this item as handled

consumers = [threading.Thread(target=consumer, args=(f"c{i}",)) for i in range(3)]
for c in consumers: c.start()

for n in range(10):                    # producer (main thread)
    q.put(n)                           # blocks if 5 items are already waiting
q.join()                               # wait until all 10 items are task_done()
print("all items processed:", sorted(processed)[:4], "...", len(processed))

for _ in consumers: q.put(None)        # one sentinel per consumer
for c in consumers: c.join()
print("consumers stopped:", not any(c.is_alive() for c in consumers))

⚠️ **Gotcha:** call `task_done()` exactly once per `get()` — including for sentinels and for items whose processing raised an exception (hence the `try/finally`). Too few calls and `q.join()` hangs forever; too many raises `ValueError`.

### `Queue.shutdown()` (Python 3.13+)

Since Python 3.13 a queue can be shut down — the closest match to `BlockingCollection.CompleteAdding()` / `ChannelWriter.Complete()`. After `shutdown()`, `put()` raises `queue.ShutDown`; `get()` keeps returning remaining items and then raises `queue.ShutDown` instead of blocking. (`shutdown(immediate=True)` discards the remaining items.) No sentinels needed:

In [ ]:
q = queue.Queue()
received = []

def drain():
    while True:
        try:
            received.append(q.get())
        except queue.ShutDown:
            return                     # queue closed and empty

t = threading.Thread(target=drain)
t.start()
for word in ["alpha", "beta", "gamma"]:
    q.put(word)
q.shutdown()                           # no more items will come
t.join()
print(received)

try:
    q.put("late")
except queue.ShutDown as e:
    print(f"{type(e).__name__} on put after shutdown")

### Non-blocking and timed operations

`get(timeout=...)` raises `queue.Empty` and `put(timeout=...)` on a full bounded queue raises `queue.Full` — like `TryTake`/`TryAdd` with a timeout. `get_nowait()`/`put_nowait()` don't wait at all.

In [ ]:
q = queue.Queue(maxsize=1)
q.put("only")
try:
    q.put_nowait("extra")
except queue.Full:
    print("queue.Full")

print(q.get_nowait())
try:
    q.get(timeout=0.05)
except queue.Empty:
    print("queue.Empty after 0.05s")

print(q.qsize(), q.empty(), q.full())   # approximate in multithreaded code!

⚠️ **Gotcha:** `qsize()`, `empty()` and `full()` are only snapshots — another thread may change the queue right after you check. Don't write `if not q.empty(): q.get()`; just `get()` with a timeout and handle `Empty`.

## `LifoQueue`, `PriorityQueue` and `SimpleQueue`

- `LifoQueue` — a thread-safe **stack** (≈ `ConcurrentStack<T>`).
- `PriorityQueue` — smallest item first (≈ a thread-safe `PriorityQueue<TElement, TPriority>`). Put `(priority, item)` tuples; add a counter as tie-breaker if items themselves aren't comparable.
- `SimpleQueue` — an unbounded FIFO without `task_done`/`join`; slightly faster.

In [ ]:
lifo = queue.LifoQueue()
for x in [1, 2, 3]:
    lifo.put(x)
print([lifo.get() for _ in range(3)])        # [3, 2, 1]

pq = queue.PriorityQueue()
pq.put((2, "send report"))
pq.put((0, "fix outage"))
pq.put((1, "review PR"))
print([pq.get()[1] for _ in range(3)])       # lowest priority number first

sq = queue.SimpleQueue()
sq.put("x")
print(sq.get())

⚠️ **Gotcha:** with `PriorityQueue`, two entries with equal priority are compared by their **second** element. If that is, say, a dict or a custom object without ordering, you get a `TypeError`. Use `(priority, sequence_number, item)` where `sequence_number` comes from `itertools.count()`.

In [ ]:
pq = queue.PriorityQueue()
pq.put((1, {"task": "a"}))
try:
    pq.put((1, {"task": "b"}))       # same priority → compares the dicts
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Mutual exclusion | `lock (obj) { }` | `with lock:` (`threading.Lock()`) |
| Reentrant lock | `lock` / `Monitor` (always reentrant) | `threading.RLock()` |
| Try-lock with timeout | `Monitor.TryEnter(obj, t)` | `lock.acquire(timeout=t)` |
| Signal | `ManualResetEventSlim.Set/Wait` | `Event.set()` / `.wait(timeout)` |
| Cancellation loop | `CancellationToken` + `Task.Delay` | `while not stop.wait(interval):` |
| Wait for condition | `Monitor.Wait/Pulse` | `Condition.wait_for(pred)` / `notify()` |
| Limit concurrency | `SemaphoreSlim(n)` | `BoundedSemaphore(n)` |
| Phase sync | `Barrier.SignalAndWait()` | `Barrier.wait()` |
| Producer/consumer | `BlockingCollection<T>`, `Channel<T>` | `queue.Queue(maxsize)` |
| Complete adding | `CompleteAdding()` / `Writer.Complete()` | sentinel or `q.shutdown()` (3.13+) |
| Wait for all work | — | `task_done()` + `q.join()` |
| Thread-safe stack / heap | `ConcurrentStack<T>` / — | `LifoQueue` / `PriorityQueue` |

## Exercises

1. Write a `BankAccount` class with `deposit(amount)` and `withdraw(amount)` methods protected by a lock. `withdraw` should raise `ValueError` if the balance is insufficient. Run 5 threads each depositing 1 a thousand times and print the final balance (5000).
2. Build a pipeline: a producer puts the words `["red", "green", "blue", "yellow"]` on a `Queue`, two consumers upper-case them into a shared list, and you stop the consumers with sentinels. Print the sorted result.
3. Use a `Semaphore(3)` to ensure that at most 3 of 8 threads are "downloading" (sleep 0.02s) at once. Record the maximum concurrency seen and print it.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import threading, queue, time

# 1 — lock protects every read-modify-write
class BankAccount:
    def __init__(self):
        self._lock = threading.Lock()
        self.balance = 0

    def deposit(self, amount):
        with self._lock:
            self.balance += amount

    def withdraw(self, amount):
        with self._lock:
            if amount > self.balance:
                raise ValueError("insufficient funds")
            self.balance -= amount

acct = BankAccount()
def add_many():
    for _ in range(1000):
        acct.deposit(1)
ts = [threading.Thread(target=add_many) for _ in range(5)]
for t in ts: t.start()
for t in ts: t.join()
print(acct.balance)                     # 5000

# 2 — producer/consumer with sentinels
q = queue.Queue()
out, out_lock = [], threading.Lock()
def consume():
    while (word := q.get()) is not None:
        with out_lock:
            out.append(word.upper())
cs = [threading.Thread(target=consume) for _ in range(2)]
for c in cs: c.start()
for w in ["red", "green", "blue", "yellow"]:
    q.put(w)
for _ in cs: q.put(None)
for c in cs: c.join()
print(sorted(out))

# 3 — semaphore caps concurrency
sem, stats = threading.Semaphore(3), threading.Lock()
active = peak = 0
def download():
    global active, peak
    with sem:
        with stats:
            active += 1
            peak = max(peak, active)
        time.sleep(0.02)
        with stats:
            active -= 1
ts = [threading.Thread(target=download) for _ in range(8)]
for t in ts: t.start()
for t in ts: t.join()
print("peak:", peak)                    # 3
```

</details>